# 03 — Image-slicer remapping to the detector frame

**Portfolio / learning project — not a calibrated instrument simulator.** All quantities below are synthetic proxies, literature-derived presets, or engineering estimates (see the README provenance table).

This notebook is narrative and self-contained; run the cells top to bottom.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
FIG = pathlib.Path.cwd().parent / 'figures' / 'generated'
FIG.mkdir(parents=True, exist_ok=True)


## From cube to detector
The image slicer cuts the field into slices, stacks them along the detector cross-dispersion axis, and disperses each spaxel into a spectrum along the columns. Each spaxel owns one detector **row**; each wavelength a **column** — which is what makes the map invertible.

In [ ]:
from ao_fed_ifu.source_cube import SourceCubeConfig, create_emission_line_cube, scale_cube_to_total_electrons
from ao_fed_ifu.spectrograph_model import SpectrographConfig
from ao_fed_ifu.ifu_mapping import IFUConfig, build_slicer_mapping, forward_project_cube

spectro = SpectrographConfig(lambda_min_um=2.0, lambda_max_um=2.3, resolving_power_R=2500.0)
cfg = SourceCubeConfig(nx=10, ny=10, nlambda=spectro.n_detector_pixels,
                       lambda_min_um=2.0, lambda_max_um=2.3)
cube = create_emission_line_cube(cfg, line_flux_rel=8.0, velocity_gradient_kms_per_spaxel=15.0)
cube = scale_cube_to_total_electrons(cube, 2e6)
ifu = IFUConfig(nx_spaxel=cfg.nx, ny_spaxel=cfg.ny, cross_talk_sigma_pix=0.7)
mapping = build_slicer_mapping(ifu, spectro, cfg.wavelength_grid_um)
print('detector shape (rows, cols):', mapping.det_shape)
mapping.spaxel_trace_table().head()

In [ ]:
frame = forward_project_cube(cube, mapping, apply_lsf=True, apply_cross_talk=True)
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(frame, aspect='auto', origin='lower', cmap='magma')
ax.set_xlabel('det_x  (dispersion / wavelength)'); ax.set_ylabel('det_y  (slices)')
ax.set_title('Image-slicer detector frame'); plt.colorbar(im, ax=ax, label='electrons')
fig.tight_layout(); fig.savefig(FIG / 'nb03_detector_frame.png', dpi=110); plt.show()

## The wavelength map
`SpectrographConfig` provides the linear wavelength solution `lambda(det_x)` and the resolution element `delta_lambda = lambda/R`.

In [ ]:
x = np.arange(spectro.n_detector_pixels)
lam = spectro.detector_x_to_wavelength(x)
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(x, lam); ax.set_xlabel('det_x [pix]'); ax.set_ylabel('wavelength [um]')
ax.set_title(f'wavelength solution (R={spectro.resolving_power_R:.0f}, '
             f'{spectro.actual_pixels_per_resolution_element():.2f} pix/resel)')
fig.tight_layout(); fig.savefig(FIG / 'nb03_wavelength_map.png', dpi=110); plt.show()
pd.DataFrame([{'det_rows': mapping.det_shape[0], 'det_cols': mapping.det_shape[1],
    'pix_per_resel': spectro.actual_pixels_per_resolution_element(),
    'well_sampled': spectro.is_well_sampled()}]).to_csv(FIG / 'nb03_summary.csv', index=False)